## CSE 655 - Deep Learning Final Project


https://drive.google.com/drive/folders/1DsUKZ1QlYlGXSEMjYSAIfu6O7Dbcy7UA?usp=drive_link 
Linki üzerinden direkt çalıştırılabilir. Test datası ilgili klasörde bulunmaktadır.

In [ ]:
!pip install -q timm groq transformers accelerate pillow scikit-learn
print("✅ Kurulumlar tamam!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 12.2 MB/s eta 0:00:00
✅ Kurulumlar tamam!


In [ ]:
CONFIG = {
    "classifier_model_path": "models\best_swin_tiny-92.0.pth,
    "groq_api_key": "YOUR_GROQ_API_KEY_HERE",
    # Groq API key: https://console.groq.com/keys adresinden alınabilir
    "test_data_dir": "data\rvlcdip_split_v3\test"
}
print("✅ Ayarlar tamam!")

✅ Ayarlar tamam!


1.  **Classifier:**  document-classification.ipynb eğittiğimiz Swin Transformer modeli

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import timm

CLASS_NAMES = ["email", "form", "invoice", "letter"]

class DocumentClassifier:
    def __init__(self, model_path):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🔍 Classification yükleniyor... (Device: {self.device})")

        # Corrected model name: 'swin_tiny_patch4_window7_224'
        self.model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=4)
        checkpoint = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()
        self.model = self.model.to(self.device)

        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        print("✅ Classification hazır!")

    def predict(self, image_path):
        image = Image.open(image_path).convert("RGB")
        image_tensor = self.transform(image).unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.model(image_tensor)
            probs = torch.softmax(outputs, dim=1)
            confidence, predicted = probs.max(1)

        return CLASS_NAMES[predicted.item()], confidence.item() * 100

# Yükle
classifier = DocumentClassifier(CONFIG["classifier_model_path"])

🔍 Classification yükleniyor... (Device: cuda)
✅ Classification hazır!


2.  **OCR Engine:** `datalab-to/chandra`

In [ ]:
import torch
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor

class ChandraOCR:
    def __init__(self):
        print("📝 Chandra OCR yükleniyor...")
        self.model = AutoModelForVision2Seq.from_pretrained(
            "datalab-to/chandra",
            torch_dtype=torch.float16,
            trust_remote_code=True,
            device_map="auto"
        ).eval()

        self.processor = AutoProcessor.from_pretrained(
            "datalab-to/chandra",
            trust_remote_code=True
        )
        print("✅ OCR hazır!")

    def extract_text(self, image_path):
        image = Image.open(image_path).convert("RGB")

        prompt = "Extract all text from this image."

        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]}]

        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text], images=[image], padding=True, return_tensors="pt")
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        with torch.no_grad():
            generated_ids = self.model.generate(**inputs, max_new_tokens=2048, do_sample=False)

        # input prompt'u kesip sadece output kısmını decode et
        trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
        out = self.processor.batch_decode(trimmed, skip_special_tokens=True)[0]

        return out.strip()

ocr = ChandraOCR()


📝 Chandra OCR yükleniyor...


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

✅ OCR hazır!


In [ ]:
import json
import re
from groq import Groq

# AYARLAR
CONFIDENCE_THRESHOLD = 70.0

# SYSTEM MESSAGE
SYSTEM_MESSAGE = """You are a document analysis assistant.
Your task is to extract information from documents and return ONLY valid JSON.

STRICT RULES:
1. Return ONLY a valid JSON object
2. No markdown, no code fences, no explanations
3. No text before or after the JSON
4. If a field is not found, DO NOT include it in the response
5. All string values must be in the same language as the document"""

# PROMPT ŞABLONLARI
PROMPTS = {
    "email": """This is an EMAIL document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: from, to, cc, date, subject, and the email summary

Text:
{ocr_text}

Return only JSON:""",

    "invoice": """This is an INVOICE document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: invoice_number, date, vendor, customer, amount, items, due_date and the invoice summary

Text:
{ocr_text}

Return only JSON:""",

    "letter": """This is a LETTER document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: from, to, date, subject, and the letter summary

Text:
{ocr_text}

Return only JSON:""",

    "form": """This is a FORM document.
Extract the information you can find from the text below.
Only include fields that you can actually find in the text.

Possible fields: form_title, organization, fields, date, purpose and the form summary

Text:
{ocr_text}

Return only JSON:""",

    "unknown": """The document type could not be determined with high confidence.
Analyze the text and provide a general summary.

Text:
{ocr_text}

Return JSON with these fields:
- topic: main topic of the document
- summary: brief summary of the content
- key_points: list of important points (max 5)

Return only JSON:"""
}

VALID_DOC_TYPES = {"email", "invoice", "letter", "form"}

class LLMAgent:
    def __init__(self, api_key):
        print("🤖 LLM Agent başlatılıyor...")
        self.client = Groq(api_key=api_key)
        self.model = "llama-3.3-70b-versatile"
        print("✅ LLM Agent hazır!")

    def _normalize_doc_type(self, doc_type):
        """Doküman türünü normalize et"""
        normalized = doc_type.lower().strip()
        if normalized in VALID_DOC_TYPES:
            return normalized
        else:
            print(f"   ⚠️ Bilinmeyen tür: {doc_type} → 'unknown'")
            return "unknown"

    def _clean_json_response(self, text):
        """JSON temizle"""
        text = re.sub(r'```json\s*', '', text)
        text = re.sub(r'```\s*', '', text)
        text = text.strip()
        if not text.startswith('{'):
            start_idx = text.find('{')
            if start_idx != -1:
                text = text[start_idx:]
        if not text.endswith('}'):
            end_idx = text.rfind('}')
            if end_idx != -1:
                text = text[:end_idx + 1]
        return text

    def analyze(self, ocr_text, doc_type, confidence):
        """
        Güven skoruna göre analiz stratejisi belirle
        """
        # 1. Normalize type
        normalized_type = self._normalize_doc_type(doc_type)

        # 2. Strateji belirleme
        if confidence < CONFIDENCE_THRESHOLD:
            print(f"   ⚠️ Düşük güven ({confidence:.1f}%) → Genel analiz")
            prompt = PROMPTS["unknown"].format(ocr_text=ocr_text)
            analysis_mode = "general"
        else:
            print(f"   ✅ Yüksek güven ({confidence:.1f}%) → {normalized_type.upper()} analizi")
            prompt = PROMPTS[normalized_type].format(ocr_text=ocr_text)
            analysis_mode = "type_specific"

        # 3. LLM çağrısı
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": SYSTEM_MESSAGE},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,
                max_tokens=1024
            )
            result_text = response.choices[0].message.content
            result_text = self._clean_json_response(result_text)

            try:
                result = json.loads(result_text)
            except json.JSONDecodeError as e:
                result = {"parse_error": str(e), "raw_response": result_text[:500]}
        except Exception as e:
            result = {"api_error": str(e)}

        # Meta bilgi
        result["_meta"] = {
            "analysis_mode": analysis_mode,
            "confidence": confidence,
            "detected_type": doc_type
        }
        return result

# Yükle
llm = LLMAgent(CONFIG["groq_api_key"])

🤖 LLM Agent başlatılıyor...
✅ LLM Agent hazır!


## 3. Pipeline Logic
Burada **Sınıflandırma** sonucuna göre **Dinamik Prompt** seçimi yapılır.

> **Mantık:** Eğer model dokümanın bir "Fatura" olduğundan %90 eminse, LLM'e *"Fatura numarasını ve toplam tutarı bul"* deriz. Eğer emin değilse, sadece *"Bu belgeyi özetle"* deriz.

In [ ]:
import os
import json

def process_document(image_path):
    """
    Tam pipeline: Classification → OCR → LLM

    Güven skoruna göre analiz stratejisi:
    - >= 70%: Türe özgü analiz (type-specific extraction)
    - < 70%: Genel analiz (generic-safe mode)
    """
    print(f"\n{'='*60}")
    print(f"📄 Dosya: {os.path.basename(image_path)}")
    print("="*60)

    # ============================================
    # ADIM 1: CLASSIFICATION
    # ============================================
    print("\n🔍 ADIM 1: Classification")
    print("-"*40)
    doc_type, confidence = classifier.predict(image_path)
    print(f"   Tahmin: {doc_type}")
    print(f"   Güven:  {confidence:.1f}%")

    # ============================================
    # ADIM 2: OCR
    # ============================================
    print("\n📝 ADIM 2: OCR (Chandra)")
    print("-"*40)
    ocr_text = ocr.extract_text(image_path)
    print(f"   Çıkarılan: {len(ocr_text)} karakter")

    # ============================================
    # ADIM 3: LLM ANALİZ
    # ============================================
    print("\n🤖 ADIM 3: LLM Analiz")
    print("-"*40)
    analysis = llm.analyze(ocr_text, doc_type, confidence)

    # Meta bilgilerini al
    meta = analysis.pop("_meta", {})

    print(f"   Mod: {meta.get('analysis_mode', 'unknown')}")

    # ============================================
    # SONUÇ
    # ============================================
    print("\n" + "="*60)
    print("✅ İşlem tamamlandı!")
    print("="*60)

    return {
        "file": os.path.basename(image_path),
        "classification": {
            "document_type": doc_type,
            "confidence": confidence
        },
        "analysis": {
            "mode": meta.get("analysis_mode", "unknown"),
            "extracted_info": analysis
        },
        "ocr_text": ocr_text
    }


def display_result(result):
    """
    Sonucu güzel formatta göster
    """
    print("\n" + "="*60)
    print("📊 SONUÇ RAPORU")
    print("="*60)

    # Classification
    print(f"\n🏷️ SINIFLANDIRMA:")
    print(f"   Tür: {result['classification']['document_type']}")
    print(f"   Güven: {result['classification']['confidence']:.1f}%")

    # Analiz modu
    mode = result['analysis']['mode']
    if mode == "type_specific":
        print(f"   Mod: ✅ Türe Özgü Analiz")
    else:
        print(f"   Mod: ⚠️ Genel Analiz (düşük güven)")

    # OCR özeti
    print(f"\n📄 OCR ÇIKTISI:")
    ocr_preview = result['ocr_text'][:300]
    if len(result['ocr_text']) > 300:
        ocr_preview += "..."
    print(f"   {ocr_preview}")

    # LLM Analizi
    print(f"\n🤖 LLM ANALİZ SONUCU:")
    print("-"*40)
    print(json.dumps(result['analysis']['extracted_info'], indent=2, ensure_ascii=False))
    print("="*60)


In [ ]:
test_image = "/content/drive/MyDrive/deep-learning/rvlcdip_split_v3/test/email/2078123093a.tif"
result = process_document(test_image)
display_result(result)


📄 Dosya: 2078123093a.tif

🔍 ADIM 1: Classification
----------------------------------------


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


   Tahmin: email
   Güven:  100.0%

📝 ADIM 2: OCR (Chandra)
----------------------------------------
   Çıkarılan: 216 karakter

🤖 ADIM 3: LLM Analiz
----------------------------------------
   ✅ Yüksek güven (100.0%) → EMAIL analizi
   Mod: type_specific

✅ İşlem tamamlandı!

📊 SONUÇ RAPORU

🏷️ SINIFLANDIRMA:
   Tür: email
   Güven: 100.0%
   Mod: ✅ Türe Özgü Analiz

📄 OCR ÇIKTISI:
   From: Lindheim, Jim(JL) on Sat, Jan 27, 1996 6:53 AM
Subject: Charlotte Sessions
To: Firesione, Marc; Purcell, Clare; Winokur, Matt
File(s): ETS/Charlotte Sessions

Please see attached memo on the Charlotte Sessions.

🤖 LLM ANALİZ SONUCU:
----------------------------------------
{
  "from": "Lindheim, Jim(JL)",
  "to": "Firesione, Marc; Purcell, Clare; Winokur, Matt",
  "date": "Sat, Jan 27, 1996 6:53 AM",
  "subject": "Charlotte Sessions"
}
